# The `acm.catalogs` module

In [ ]:
# Mock imports - Run ths cell to avoid import errors in the test notebook
import numpy as np
import pandas as pd

from acm.catalogs.backends import SnapshotBackend
from acm.catalogs.dataclasses import Tracer


class CosmologyPlaceholder:
    """A placeholder cosmology class for the example."""

    def efunc(self, z) -> float:  # noqa: ANN001, ARG002, D102
        return 1.0  # Placeholder for the expansion function

    def angular_diameter_distance(self, z) -> float:  # noqa: ANN001, ARG002, D102
        return 1000.0  # Placeholder for angular diameter distance


def tracer_data(n: int = 100) -> pd.DataFrame:
    """Generate minimal valid tracer data."""
    rng = np.random.default_rng()
    return pd.DataFrame({
        "x": rng.uniform(0, 100, n),
        "y": rng.uniform(0, 100, n),
        "z": rng.uniform(0, 100, n),
        "vx": rng.normal(0, 1, n),
        "vy": rng.normal(0, 1, n),
        "vz": rng.normal(0, 1, n),
    })

class DummyBackend(SnapshotBackend):
    """A minimal implementation of DarkMatterBackend for testing."""

    def load_dark_matter_catalog(self, redshift: float, **kwargs) -> None:  # noqa: ARG002, D102
        return None  # Return a dummy catalog object

    def get_dark_matter_catalog(self, redshift: float) -> None:  # noqa: ARG002, D102
        return None  # Return a dummy catalog object

    def make_galaxy_catalog(self, dm_catalog, tracers: list[Tracer], **kwargs) -> dict:  # noqa: ANN001, ARG002, D102
        return {tracer: tracer_data() for tracer in tracers}

    @property
    def boxsize(self) -> float:  # noqa: D102
        return 500.0

### 1. `acm.catalogs.dataclasses`

This module registers two dataclasses: 
- `Tracer` registers a name and parameters used by the Dark Matter backends (such as HOD parameters)
- `Transform` registers a name and a function with extra parameters. Can also register a tracer name (str) to apply this transform to the specified tracer only.

In [ ]:
from acm.catalogs.dataclasses import Tracer, Transform

# register a tracer
tracer_LRG = Tracer(name="LRG", params={"color": "red", "logMcut": 13.5})  # noqa: N816

# Register a transform, for example, a simple RSD transformation
def _rsd(data: pd.DataFrame, z: float) -> pd.DataFrame:
    """Apply a simple RSD transformation to the data."""
    cosmo = CosmologyPlaceholder()
    f = cosmo.efunc(z)  # Growth rate placeholder
    data["z"] += data["vz"] * f / cosmo.angular_diameter_distance(z)  # Simplified RSD effect
    return data

transform_rsd = Transform(name="RSD", func=_rsd, kwargs={"z": 0.5})

# The transform can be applied to the tracer data as follows:
data = tracer_data(n=1000)
data_rsd = transform_rsd.apply(data)

data_rsd.head()

### 2. `acm.catalogs.products`
This module registers galaxy catalog classes, that allow to load several tracers. The `SnapshotGalaxyCatalog` class implements the galaxy catalogs for cubic boxes.

In [ ]:
from acm.catalogs.dataclasses import Tracer
from acm.catalogs.products import SnapshotCatalog

tracer_LRG = Tracer(name="LRG", params={"color": "red", "logMcut": 13.5})  # noqa: N816
tracer_ELG = Tracer(name="ELG", params={"color": "blue", "logMcut": 12.0})  # noqa: N816

catalog = SnapshotCatalog(
    redshift=0.5,
    cosmo = CosmologyPlaceholder(),  # Placeholder for a cosmology object  # ty:ignore[invalid-argument-type]
    cosmo_fid= CosmologyPlaceholder(),  # Placeholder for a fiducial cosmology object  # ty:ignore[invalid-argument-type]
    boxsize=[100, 100, 100],
)
catalog.set_tracer_data(tracer_LRG, tracer_data(n=10000))
catalog.set_tracer_data(tracer_ELG, tracer_data(n=1000))
catalog

In [ ]:
# pre-defined transforms can be added to the catalog
catalog.ap(los='z')
catalog.rsd(los='z')
catalog.downsample('LRG', f_gal=0.1, seed=42)

# Try to run the cell several times: transforms are applied at get_tracer

catalog.get_tracer_data('LRG', raw=False).head() # raw=True will return the original data without transforms applied

In [ ]:
# You can check the order of the transforms by inspecting:
print(catalog.transform_pipeline)

# Transforms are strored internally as:
print(catalog._transforms)

In [ ]:
# We can add an external transform directly to the catalog as well:
catalog._add_transform(transform_rsd)

catalog._transforms

In [ ]:
from acm.catalogs.products import RandomSnapshotCatalog

# Build a random catalog from the existing snapshot catalog
random_catalog = RandomSnapshotCatalog.from_snapshot(catalog, seed=42)

print('Box size of random catalog:', random_catalog.boxsize)

random_catalog.get_tracer_data('LRG').head()

### 3. `acm.catalogs.backends`
Backends are implemented to allow the separation between the galaxy catalog creation process and the galaxy catalogs operations. The current implementation assumes dark matter catalog processes, with the `DarkmatterBackend` class. Its subclasses implement `get_dark_matter_catalog` to load the catalogs on disk, and `make_galaxy_catalog` to create galaxy catalogs.

### 4. `acm.catalogs.factories`
Factories implement backends to create galaxy catalogs


In [ ]:

# testing factories
from acm.catalogs.factories.snapshot import SnapshotCatalogFactory

backend = DummyBackend() # Can be an instance of a backend or a string for a registered backend
backend.load_dark_matter_catalog(redshift=0.5)  # Load and cache the dark matter catalog for the specified redshift

factory = SnapshotCatalogFactory(
    backend=backend,
    catalog_class=SnapshotCatalog,
    cosmo=CosmologyPlaceholder(),  # ty:ignore[invalid-argument-type]
    cosmo_fid=CosmologyPlaceholder(),  # ty:ignore[invalid-argument-type]
)

tracer_LRG = Tracer(name="LRG", params={"color": "red", "logMcut": 13.5})  # noqa: N816
tracer_ELG = Tracer(name="ELG", params={"color": "blue", "logMcut": 12.0})  # noqa: N816

factory.make_catalogs(redshifts=[0.5], tracers=[tracer_LRG, tracer_ELG])

catalog = factory.get_catalog(0.5)
catalog.ap(los='z')
catalog.rsd(los='z')

catalog['ELG'].head() # get_tracer_data can be accessed directly via the catalog reference

In [ ]:
# NOTE: transforms are stored in the catalog reference, so the get_catalog method will return the same catalog with transforms applied at get_tracer_data
factory.get_catalog(0.5)['ELG'].head()